# Semantic Textual Similarity - Final Project

##### Authors: Josep de Cid and Gonzalo Recio


value |	constraint
:--- | --- 
10 |     if the pearson is over 10th participant (**.7562**)
0  |	  if the pearson is under the baseline (**.311**) 


In [6]:
import nltk

from nltk import pos_tag
from nltk.metrics import jaccard_distance
from nltk.stem import WordNetLemmatizer
import numpy as np

from scipy.stats import pearsonr

nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /home/gonzalo/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/gonzalo/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to /home/gonzalo/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

Read datasets utils

In [7]:

def read_dataset(path):
    sentences, scores = [], []
    with open(path, mode='r') as f:
        for idx, line in enumerate(f.readlines()):
            a, b = line.strip().split('\t')
            sentences.append([a, b])
            # print(f'{idx + 1}a) {a}')
            # print(f'{idx + 1}b) {b}\n')
    gs_path = path.split('.')
    gs_path[1] = 'gs'
    gs_path = '.'.join(gs_path)
    with open(gs_path, 'r') as f:
        scores = [float(line.strip()) for line in f.readlines()]
    return sentences, scores

In [8]:
def read_all_dataset(base_path = 'test-gold/'):
    datasets_test = "STS.input.MSRpar.txt", "STS.input.MSRvid.txt", "STS.input.SMTeuroparl.txt", \
                "STS.input.surprise.OnWN.txt", "STS.input.surprise.SMTnews.txt"
    datasets_train = "STS.input.MSRpar.txt", "STS.input.MSRvid.txt", "STS.input.SMTeuroparl.txt"
    if 'test' in base_path:
        datasets = datasets_test
    else:
        datasets = datasets_train
    sentences_ = []
    all_scores_ = []
    for dataset in datasets:
        sentences, scores = read_dataset(base_path + dataset)
        sentences_  += (sentences)
        all_scores_ += scores
        
    all_scores = []
    if 'test' in base_path:
        with open(base_path + 'STS.gs.ALL.txt', mode='r') as f:
            all_scores = [float(line.strip()) for line in f.readlines()]
        assert all_scores == all_scores_, "Datasets not collected in correct order"
    
    return sentences_, all_scores_

In [9]:
pos_translation = {
    'NN': 'n',
    'NNP': 'n',
    'NNS': 'n',
    'NNPS': 'n',
    'VB': 'v',
    'VBD': 'v',
    'VBG': 'v',
    'VBN': 'v',
    'VBP': 'v',
    'VBZ': 'v',
    'JJ': 'a',
    'JJR': 'a',
    'JJS': 'a',
    'RB': 'r',
    'RBR': 'r',
    'RBS': 'r'   
}

First approach

In [10]:
sentences, scores = read_all_dataset()

In [11]:
def tag_sentence(sentence, lang='english'):
    return pos_tag(nltk.word_tokenize(sentence, language=lang))

def tag_pair_sentences(pair, lang):
    return (
        pos_tag(nltk.word_tokenize(pair[0], language=lang)),
        pos_tag(nltk.word_tokenize(pair[1], language=lang))
    )

wnl = WordNetLemmatizer()

def lemmatize_token(token):
    if token[1][0] in pos_translation:
        return wnl.lemmatize(token[0].lower(), pos=pos_translation[token[1][0]])
    return token[0]

def lemmatize_sentence(sentence):
    return list(map(lemmatize_token, sentence))

def lemmatize_pair_sentences(pair):
    return (
        lemmatize_sentence(pair[0]),
        lemmatize_sentence(pair[1])
    )

In [12]:
def jaccard_similarity_sentences(pair, lang='english'):
    #pair = tag_pair_sentences(pair, lang)
    #pair = lemmatize_pair_sentences(pair)
    return 1-jaccard_distance(set(pair[0]), set(pair[1]))

In [13]:
! pip install gensim

You should consider upgrading via the 'pip install --upgrade pip' command.


In [16]:
flatten = lambda l: [item for sublist in l for item in sublist]
train_sentences = flatten(X_train)

In [17]:
import numpy as np
REAL = np.float32 

def sif_embeddings(sentences, model, alpha=1e-3):
    """Compute the SIF embeddings for a list of sentences
    Parameters
    ----------
    sentences : list
        The sentences to compute the embeddings for
    model : `~gensim.models.base_any2vec.BaseAny2VecModel`
        A gensim model that contains the word vectors and the vocabulary
    alpha : float, optional
        Parameter which is used to weigh each individual word based on its probability p(w).
    Returns
    -------
    numpy.ndarray 
        SIF sentence embedding matrix of dim len(sentences) * dimension
    """
    
    vlookup = model.wv.vocab  # Gives us access to word index and count
    vectors = model.wv        # Gives us access to word vectors
    size = model.vector_size  # Embedding size
    
    Z = 0
    for k in vlookup:
        Z += vlookup[k].count # Compute the normalization constant Z
    
    output = []
    
    # Iterate all sentences
    for s in sentences:
        count = 0
        v = np.zeros(size, dtype=REAL) # Summary vector
        # Iterare all words
        for w in s:
            # A word must be present in the vocabulary
            if w in vlookup:
                for i in range(size):
                    v[i] += ( alpha / (alpha + (vlookup[w].count / Z))) * vectors[w][i]
                count += 1 
                
        if count > 0:
            for i in range(size):
                v[i] *= 1/count
        output.append(v)
    return np.vstack(output).astype(REAL)

In [25]:
from gensim.models import Word2Vec
sentences = [['cat', 'say', 'meow'], ['dog', 'say', 'woof']]
# model = Word2Vec(X_train, min_count=1)
# print(model.wv.vocab['cat'])
# def tokenized_pairs
tokenized = [lemmatize_sentence(tag_sentence(s)) for s in train_sentences]
# tokenized
# model = Word2Vec(tokenized, min_count=1)
# model = Word2Vec()
# tokenized = [nltk.word_tokenize(s, language='english') for s in train_sentences]
model = Word2Vec(tokenized)

TypeError: expected string or bytes-like object

In [ ]:
# print(model.wv.vocab['cat'])

In [26]:
from sklearn.neural_network import MLPRegressor

X_train, y_train = read_all_dataset(base_path='train/')
pairs = [tag_pair_sentences(pair, lang='english') for pair in X_train]
X_train = [lemmatize_pair_sentences(pair) for pair in pairs]

mlp = MLPRegressor(hidden_layer_sizes=(100,50,10,),
                       activation='relu',
                       solver='adam',
                       learning_rate='adaptive',
                       max_iter=1000,
                       learning_rate_init=0.01,
                       alpha=0.01,
                       early_stopping=True)
# model = gensim.models.Word2Vec()

In [20]:
# X_train = list(map(X_train))
# X_train_dist = np.array(list(map(jaccard_similarity_sentences, X_train)))
sents_emb = np.array([sif_embeddings(sents, model) for sents in X_train])
m = min(y_train)
M = max(y_train)


NameError: name 'model' is not defined

In [ ]:
sents_emb2.shape

In [ ]:
sents_emb2 = np.array([np.append(s[0],s[1]) for s in sents_emb])

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
sents_emb[0][1]
a = [s[0] for s in sents_emb]
b = [s[1] for s in sents_emb]
cosine_similarity([sents_emb[4][0]], [sents_emb[4][1]])
# y_train[4]

In [ ]:
mlp.fit(sents_emb2.astype(np.float64), y_train)

In [ ]:
X_test, y_test = read_all_dataset()
pairs = [tag_pair_sentences(pair, lang='english') for pair in X_test]
X_test = [lemmatize_pair_sentences(pair) for pair in pairs]
X_test_transformed = [sif_embeddings(sents, model) for sents in X_test]
sents_emb2_test = np.array([ np.append(s[0],s[1]) for s in X_test_transformed])
res = mlp.predict(sents_emb2_test)

In [ ]:
corr = pearsonr(y_test, res)[0]
print(f'Correlation: {corr}  Nota: {(corr-0.311)/(0.7562-0.311)*10}')

In [ ]:
similarities = list(map(jaccard_similarity_sentences, sentences))



corr = pearsonr(scores, similarities)[0]
print(f'Correlation: {corr}  Nota: {(corr-0.311)/(0.7562-0.311)*10}')

In [36]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

corpus = [
     'This is the first document.',
     'This document is the second document.',
     'And this is the third one.',
     'Is this the first document?',
]
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(corpus)

a = vectorizer.transform(['this is a document'])
b = vectorizer.transform(['this is the document'])
cosine_similarity(a,b)[0][0]

  (0, 8)	0.5348220568369216
  (0, 3)	0.5348220568369216
  (0, 1)	0.6541641499206824


0.8818072556017398

In [39]:
lematized_copus = [[[('a','NN'),('b','NN')],[('c','NN'), ('d','NN'), ('e','NN')]],
                   [[('a','NN'),('b','NN')],[('c','NN'), ('d','NN'), ('e','NN')]],
                  ]
for p in lematized_copus:
    print(p[0],p[1])

[for p in lematized_copus]

[('a', 'NN'), ('b', 'NN')] [('c', 'NN'), ('d', 'NN'), ('e', 'NN')]
[('a', 'NN'), ('b', 'NN')] [('c', 'NN'), ('d', 'NN'), ('e', 'NN')]


2-grams


In [72]:
from nltk import ngrams, edit_distance, edit_distance_align, jaccard_distance
def jaccard_dist_ngrams(s1,s2):
    s1n, s2n = set(ngrams(s1,3)), set(ngrams(s2,3))
    print(s1n)
    print(s2n)
    return jaccard_distance(set(s1n), set(s2n))

jaccard_dist_ngrams('hey boee', 'hi boy')
# ' '.join(words[1])


' '.join(['a', 'b', 'c'])

{(' ', 'b', 'o'), ('e', 'y', ' '), ('y', ' ', 'b'), ('o', 'e', 'e'), ('h', 'e', 'y'), ('b', 'o', 'e')}
{('h', 'i', ' '), (' ', 'b', 'o'), ('i', ' ', 'b'), ('b', 'o', 'y')}


'a b c'